# Create per-variable time-series files from CAM h0 / h0a history

Concatenates Jan–Dec monthly CAM history files across a range of years into single-variable time-series files, one file per variable per case. Supports multiple cases/simulations.

Output naming (matches asphilli/NCL convention):

```
<case>.cam.<hist_pref>.<VAR>.<SYEAR>01-<EYEAR><EMONTH>.nc
```

Uses `ncrcat` (NCO) under the hood for speed — same approach as the reference NCL script.

In [62]:
import os
import glob
import re
import subprocess
import shutil
from pathlib import Path

## User configuration

Set the cases, year range, history stream, and variables to extract. Each case is a dict so different simulations can live under different users / archive roots.

Notes:
- `syear` / `eyear` are 4-character strings (zero-padded), matching the NCL script.
- `hist_pref` should be `h0` for standard CAM history, `h0a` for the newer averaged stream.
- `run_parallel = True` launches one `ncrcat` per variable in the background (like the `&` in the NCL script). Set to `False` for serial runs.
- `overwrite = False` skips a variable if its output file already exists.

In [63]:
# ---- Year range (strings, 4 chars) ----
syear  = "1979"
eyear  = "2002"
emonth = "12"     # last month of eyear to include in output filename

# ---- History stream ----
hist_pref = "h0a"          # "h0" or "h0a"

# ---- Variables to extract (one output file per variable per case) ----
atm_vars = ["PSL", "TS", "TREFHT", "PRECC", "PRECL"]

# Extra coordinate/aux vars to always keep (coLmma-separated for ncrcat -v).
# Empty string means: extract only the target variable + its own coords.
# Assumes FV grid.
extra_vars = "lat,lon,hyam,hybm,hyai,hybi"

# ---- Cases / simulations ----
# Each entry:
#   name    : casename (used in input glob + output filenames)
#   in_root : archive root; input dir is <in_root>/<name>/atm/hist/
#   out_root: output root; output dir is <out_root>/<name>/tseries/
# CAM-ZMKE cases
# f.e22.FHIST.f09_f09.CAM6.L32.000
# f.e30.FHIST.CAM6-phys.L32.000a

# f.e30.FLTHIST.CAM7.L32.000a
# f.e30.FLTHIST.CAM7.L58.000a

# (f.e30.FLTHIST.CAM7.L48.000a)

# f.e30.FLTHIST.CAM7.L32.001a
# f.e30.FLTHIST.CAM7.L48.001a
# f.e30.FLTHIST.CAM7.L58.001a
# 

cases = [
    {
        "name":     "f.e30.FLTHIST.CAM7.L58.000a",
        "in_root":  "/glade/derecho/scratch/rneale/archive/",
        "out_root": "/glade/derecho/scratch/rneale/archive/",
    },
    # Add more cases here, e.g.:
    # {
    #     "name":     "b.e30_beta05.BLT1850.ne30_t232_wgx3.127",
    #     "in_root":  "/glade/derecho/scratch/gmarques/archive",
    #     "out_root": "/glade/derecho/scratch/rneale/archive",
    # },
]

# ---- Behavior ----
run_parallel   = True     # launch ncrcat jobs in background per variable
overwrite      = True    # if False, skip variables whose output file exists
check_filecount = True    # verify #files == (eyear-syear+1)*12 before running
dry_run        = False    # if True, only print commands, do not execute

In [64]:
# Sanity check: ncrcat must be on PATH.
ncrcat = shutil.which("ncrcat")
if ncrcat is None:
    raise RuntimeError("ncrcat (NCO) not found on PATH. On Derecho/Casper: `module load nco`.")
print(f"Using ncrcat: {ncrcat}")

Using ncrcat: /glade/u/apps/opt/miniforge/envs/npl-2026a/bin/ncrcat


In [65]:
def collect_files(indir, hist_pref, syear, eyear):
    """Return sorted list of monthly h0/h0a files whose YYYY tag is within [syear, eyear].

    Files are expected to look like <case>.cam.<hist_pref>.<YYYY>-<MM>.nc
    """
    pattern = os.path.join(indir, f"*.cam.{hist_pref}.????-??.nc")
    all_files = sorted(glob.glob(pattern))

    sy, ey = int(syear), int(eyear)
    tag = re.compile(rf"\.{re.escape(hist_pref)}\.(\d{{4}})-(\d{{2}})\.nc$")

    kept = []
    for f in all_files:
        m = tag.search(f)
        if not m:
            continue
        yr = int(m.group(1))
        if sy <= yr <= ey:
            kept.append(f)
    return kept


def expected_count(syear, eyear):
    return (int(eyear) - int(syear) + 1) * 12


def run_cmd(cmd, background=False, verbose=True):
    """Execute a shell command. Returns Popen if background, else CompletedProcess."""
    if verbose:
        print("  $", cmd)
    if dry_run:
        return None
    if background:
        return subprocess.Popen(cmd, shell=True)
    return subprocess.run(cmd, shell=True, check=True)

## Generate the time series

For each case, verify the file count against the requested year range, then launch one `ncrcat` per variable. If `run_parallel = True`, the notebook waits for all jobs from that case to finish before moving to the next.

In [66]:
for case in cases:
    name     = case["name"]
    indir    = os.path.join(case["in_root"],  name, "atm", "hist") + "/"
    outdir   = os.path.join(case["out_root"], name, "tseries")    + "/"

    print("=" * 78)
    print(f"CASE   : {name}")
    print(f"indir  : {indir}")
    print(f"outdir : {outdir}")
    print(f"stream : {hist_pref}   years: {syear}-{eyear}")

    Path(outdir).mkdir(parents=True, exist_ok=True)

    files = collect_files(indir, hist_pref, syear, eyear)
    nfiles = len(files)
    nexp   = expected_count(syear, eyear)
    print(f"files found in range: {nfiles}  (expected {nexp})")
    if nfiles:
        print(f"  first: {os.path.basename(files[0])}")
        print(f"  last : {os.path.basename(files[-1])}")

    if nfiles == 0:
        print("  !! No files found — skipping this case.")
        continue

    if check_filecount and nfiles != nexp:
        print("  !! File count mismatch — skipping this case (set check_filecount=False to override).")
        continue

    # Build variable list for ncrcat -v (target var plus any extras)
    procs = []
    for var in atm_vars:
        var_list = var if not extra_vars else f"{var},{extra_vars}"
        outfile  = f"{outdir}{name}.cam.{hist_pref}.{var}.{syear}01-{eyear}{emonth}.nc"

        if (not overwrite) and os.path.exists(outfile):
            print(f"  [skip] exists: {outfile}")
            continue

        # Pass the *filtered* file list to ncrcat, not a directory glob — otherwise any
        # files outside [syear, eyear] present in the hist dir would also be concatenated,
        # producing a file whose contents don't match its 197901-200212 suffix.
        in_files = " ".join(files)
        cmd = f"ncrcat -h -O -v {var_list} {in_files} {outfile}"

        print(f"\n  -- {var} -> {os.path.basename(outfile)}")
        # Print a short summary of the command (full file list would be huge).
        print(f"     ncrcat -h -O -v {var_list} <{nfiles} files: {os.path.basename(files[0])} .. {os.path.basename(files[-1])}> -> {os.path.basename(outfile)}")
        p = run_cmd(cmd, background=run_parallel, verbose=False)
        if run_parallel and p is not None:
            procs.append((var, p))

    # Wait for this case's background jobs before starting the next case.
    if run_parallel and procs:
        print(f"\n  Waiting on {len(procs)} background ncrcat jobs for case {name} ...")
        for var, p in procs:
            rc = p.wait()
            status = "OK" if rc == 0 else f"FAIL (rc={rc})"
            print(f"    {var:12s}  {status}")

print("\n---- ALL CASES DONE ----")

CASE   : f.e30.FLTHIST.CAM7.L58.000a
indir  : /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L58.000a/atm/hist/
outdir : /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L58.000a/tseries/
stream : h0a   years: 1979-2002
files found in range: 288  (expected 288)
  first: f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.1979-01.nc
  last : f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.2002-12.nc

  -- PSL -> f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.PSL.197901-200212.nc
     ncrcat -h -O -v PSL,lat,lon,hyam,hybm,hyai,hybi <288 files: f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.1979-01.nc .. f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.2002-12.nc> -> f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.PSL.197901-200212.nc

  -- TS -> f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.TS.197901-200212.nc
     ncrcat -h -O -v TS,lat,lon,hyam,hybm,hyai,hybi <288 files: f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.1979-01.nc .. f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.2002-12.nc> -> f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.TS.197901-200212.nc

  -- TREFHT -> f.e30.FLTH

## Optional: quick verification

List the produced files and their time dimension length for the first case.

In [67]:
import xarray as xr

case  = cases[0]
outdir = os.path.join(case["out_root"], case["name"], "tseries") + "/"
produced = sorted(glob.glob(f"{outdir}{case['name']}.cam.{hist_pref}.*.{syear}01-{eyear}{emonth}.nc"))

print(f"Produced {len(produced)} file(s) in {outdir}")
for f in produced:
    try:
        with xr.open_dataset(f, decode_times=False) as ds:
            nt = ds.sizes.get("time", "n/a")
            print(f"  {os.path.basename(f):70s}  ntime={nt}")
    except Exception as e:
        print(f"  {os.path.basename(f):70s}  !! {e}")

Produced 5 file(s) in /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L58.000a/tseries/
  f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.PRECC.197901-200212.nc              ntime=288
  f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.PRECL.197901-200212.nc              ntime=288
  f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.PSL.197901-200212.nc                ntime=288
  f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.TREFHT.197901-200212.nc             ntime=288
  f.e30.FLTHIST.CAM7.L58.000a.cam.h0a.TS.197901-200212.nc                 ntime=288
